In [ ]:
# ---------------- Imports ----------------
import os
import json
import re

import yaml
import pandas as pd



In [ ]:
# ---------------- Inputs ----------------
argument_files = ["train.csv", "dev.csv", "test.csv"]

output_file = "ibm-rank-30k.jsonl"



In [ ]:
# ---------------- Config ----------------
with open("../../config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

data_folder = os.path.join(config["paths"]["proj-store"], "data")

# Input and output folder paths
input_folder = os.path.join(data_folder, "ibm_rank_30k", "raw")

output_folder = os.path.join(data_folder, "ibm_rank_30k", "processed")
os.makedirs(output_folder, exist_ok=True)
output_path = os.path.join(output_folder, output_file)

output_stem = os.path.splitext(output_file)[0]
topics_file = f"{output_stem}-topics.jsonl"
topics_path = os.path.join(output_folder, topics_file)



In [ ]:
# ---------------- Load arguments and write JSONL ----------------
exclude_columns = ["MACE-P", "stance_WA", "stance_WA_conf"]

count = 0
topics = set()

with open(output_path, "w", encoding="utf-8") as f:

    for file_name in argument_files:
        file_path = os.path.join(input_folder, file_name)

        df = pd.read_csv(file_path)

        # Remove unused columns
        df = df.drop(
            columns=exclude_columns,
            errors="ignore"
        )

        # Track source split
        df["source_file"] = file_name

        for record in df.to_dict(orient="records"):

            # Replace NaN with None
            record = {
                key: None if pd.isna(value) else value
                for key, value in record.items()
            }

            # Collect unique topics
            if record.get("topic") is not None:
                topics.add(record["topic"])

            f.write(json.dumps(record, ensure_ascii=False) + "\n")

            count += 1



In [ ]:
# ---------------- Main ----------------
topics = sorted(topics)

with open(topics_path, "w", encoding="utf-8") as f:
    for topic in topics:
        record = {"topic": topic}
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Wrote {count:,} records to {output_path}")
print(f"Wrote {len(topics):,} unique topics to {topics_path}")

